In [62]:
import polars as pl
import pandas as pd
import bottleneck as bn
import numpy as np

from google.colab import drive
drive.mount('/content/gdrive')

%cd gdrive/MyDrive/ITMO_HW/

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
[Errno 2] No such file or directory: 'gdrive/MyDrive/ITMO_HW/'
/content/gdrive/MyDrive/ITMO_HW


In [4]:
#считываем датафрейм с помощью polars
df_pl = pl.read_csv("train.csv")

In [14]:
#основная информация о датасете - первые 5 строк
df_pl.head()

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str
1,0,3,"""Braund, Mr. Owen Harris""","""male""",22.0,1,0,"""A/5 21171""",7.25,null,"""S"""
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",38.0,1,0,"""PC 17599""",71.2833,"""C85""","""C"""
3,1,3,"""Heikkinen, Miss. Laina""","""female""",26.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S"""
4,1,1,"""Futrelle, Mrs. Jacques Heath (…","""female""",35.0,1,0,"""113803""",53.1,"""C123""","""S"""
5,0,3,"""Allen, Mr. William Henry""","""male""",35.0,0,0,"""373450""",8.05,null,"""S"""


In [19]:
#основная информация о датасете - кол-во строк и столбцов
df_pl.shape

(891, 12)

In [22]:
#основная информация о датасете - размер памяти
df_pl.estimated_size()

85871

In [9]:
#основная информация о датасете - статистика по столбцам
df_pl.describe()

statistic,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
str,f64,f64,f64,str,str,f64,f64,f64,str,f64,str,str
"""count""",891.0,891.0,891.0,"""891""","""891""",714.0,891.0,891.0,"""891""",891.0,"""204""","""889"""
"""null_count""",0.0,0.0,0.0,"""0""","""0""",177.0,0.0,0.0,"""0""",0.0,"""687""","""2"""
"""mean""",446.0,0.383838,2.308642,null,null,29.699118,0.523008,0.381594,null,32.204208,null,null
"""std""",257.353842,0.486592,0.836071,null,null,14.526497,1.102743,0.806057,null,49.693429,null,null
"""min""",1.0,0.0,1.0,"""Abbing, Mr. Anthony""","""female""",0.42,0.0,0.0,"""110152""",0.0,"""A10""","""C"""
"""25%""",224.0,0.0,2.0,null,null,20.0,0.0,0.0,null,7.925,null,null
"""50%""",446.0,0.0,3.0,null,null,28.0,0.0,0.0,null,14.4542,null,null
"""75%""",669.0,1.0,3.0,null,null,38.0,1.0,0.0,null,31.0,null,null
"""max""",891.0,1.0,3.0,"""van Melkebeke, Mr. Philemon""","""male""",80.0,8.0,6.0,"""WE/P 5735""",512.3292,"""T""","""S"""


In [13]:
#основная информация о датасете - типы данных
df_pl.dtypes

[Int64,
 Int64,
 Int64,
 String,
 String,
 Float64,
 Int64,
 Int64,
 String,
 Float64,
 String,
 String]

In [15]:
#основная информация о датасете - кол-во null в столбцах
df_pl.null_count()

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,177,0,0,0,0,687,2


In [16]:
#основная информация о датасете - средние значения по столбцам (эти данные есть в describe(), добавил отдельно mean т.к. есть в ТЗ)
df_pl.mean()

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
f64,f64,f64,str,str,f64,f64,f64,str,f64,str,str
446.0,0.383838,2.308642,null,null,29.699118,0.523008,0.381594,null,32.204208,null,null


In [24]:
#Посчитайте количество пассажиров каждого класса (Pclass)
df_pl.get_column("Pclass").value_counts()

Pclass,count
i64,u32
3,491
2,184
1,216


In [26]:
#Выведите количество выживших мужчин и женщин на корабле
df_pl.group_by("Sex").agg(pl.col("Survived").sum())

Sex,Survived
str,i64
"""male""",109
"""female""",233


In [103]:
#Выведите часть таблицы с пассажирами, возраст которых больше 44 лет
df_pl.filter(pl.col("Age") > 44)

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str
7,0,1,"""McCarthy, Mr. Timothy J""","""male""",54.0,0,0,"""17463""",51.8625,"""E46""","""S"""
12,1,1,"""Bonnell, Miss. Elizabeth""","""female""",58.0,0,0,"""113783""",26.55,"""C103""","""S"""
16,1,2,"""Hewlett, Mrs. (Mary D Kingcome…","""female""",55.0,0,0,"""248706""",16.0,null,"""S"""
34,0,2,"""Wheadon, Mr. Edward H""","""male""",66.0,0,0,"""C.A. 24579""",10.5,null,"""S"""
53,1,1,"""Harper, Mrs. Henry Sleeper (My…","""female""",49.0,1,0,"""PC 17572""",76.7292,"""D33""","""C"""
…,…,…,…,…,…,…,…,…,…,…,…
858,1,1,"""Daly, Mr. Peter Denis ""","""male""",51.0,0,0,"""113055""",26.55,"""E17""","""S"""
863,1,1,"""Swift, Mrs. Frederick Joel (Ma…","""female""",48.0,0,0,"""17466""",25.9292,"""D17""","""S"""
872,1,1,"""Beckwith, Mrs. Richard Leonard…","""female""",47.0,1,1,"""11751""",52.5542,"""D35""","""S"""


In [28]:
#Считайте датасет с помощью pandas
df_pd = pd.read_csv('train.csv')

In [33]:
#Посчитайте средний возраст пассажиров с помощью bottleneck
bn.nanmean(df_pd['Age'])

29.69911764705882

In [34]:
#Посчитайте стандартное отклонение возраста пассажиров с помощью bottleneck
bn.nanstd(df_pd['Age'])

14.516321150817317

In [42]:
#Для каждого пассажира умножьте значение столбца Fare на 1.3 и сохраните результаты как новый столбец Fare_new
df_pd['Fare_new'] = df_pd['Fare'].apply(lambda row: row * 1.3)
df_pd.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Fare_new
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,9.42500
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,92.66829
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,10.30250
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,69.03000
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,10.46500


In [92]:
#Считайте датасет из файла Housing.csv (это данные о ценах домов) с помощью pandas
#также сразу смотрим первые 5 строк
df_pd = pd.read_csv('Housing.csv')
df_pd.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


**Для каждого столбца определите оптимальный с точки зрения потребления памяти тип данных**

In [93]:
#для начала смотрим текущие типы данных
df_pd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   price             545 non-null    int64 
 1   area              545 non-null    int64 
 2   bedrooms          545 non-null    int64 
 3   bathrooms         545 non-null    int64 
 4   stories           545 non-null    int64 
 5   mainroad          545 non-null    object
 6   guestroom         545 non-null    object
 7   basement          545 non-null    object
 8   hotwaterheating   545 non-null    object
 9   airconditioning   545 non-null    object
 10  parking           545 non-null    int64 
 11  prefarea          545 non-null    object
 12  furnishingstatus  545 non-null    object
dtypes: int64(6), object(7)
memory usage: 55.5+ KB


In [94]:
#смотрим статистику по данным int64, из ключевого смотрим на минимальные и максимальные значения, для определения допустимого диапазона и возможных типов данных
df_pd.describe()

,price,area,bedrooms,bathrooms,stories,parking
count,5.450000e+02,545.000000,545.000000,545.000000,545.000000,545.000000
mean,4.766729e+06,5150.541284,2.965138,1.286239,1.805505,0.693578
std,1.870440e+06,2170.141023,0.738064,0.502470,0.867492,0.861586
min,1.750000e+06,1650.000000,1.000000,1.000000,1.000000,0.000000
25%,3.430000e+06,3600.000000,2.000000,1.000000,1.000000,0.000000
50%,4.340000e+06,4600.000000,3.000000,1.000000,2.000000,0.000000
75%,5.740000e+06,6360.000000,3.000000,2.000000,2.000000,1.000000
max,1.330000e+07,16200.000000,6.000000,4.000000,4.000000,3.000000


Зная диапазоны  
```
int8: -128 - 127
int16: -32768 - 32767
int32: -2.1*10^9 - 2.1*10^9
int64: -9.2*10^18 - 9.2*10^18
uint8: 0 - 255
uint16: 0 - 65535
uint32: 0 - 4.2*10^9
uint64: 0 - 1.8*10^19
```
Выбираем минимально допустимые типы (меньшего размера):  
price - int32 либо uint32  
area - int16 либо uint16  
bedrooms - int8 либо uint8  
bathrooms - int8 либо uint8  
stories - int8 либо uint8  
parking - int8 либо uint8

In [61]:
#далее смотрим на оставшиеся столбцы с типом object, чтобы оценить профит от перехода к категориальным значениям
#(если уникальных значений мало - профит высокий)
#здесь видим, что по всем столбцам уникальных значений 2 и 3 - высокий профит от перехода к категориальным значениям
df_pd[["mainroad", "guestroom", "basement", "hotwaterheating", "airconditioning", "prefarea", "furnishingstatus"]].nunique()

,0
mainroad,2
guestroom,2
basement,2
hotwaterheating,2
airconditioning,2
prefarea,2
furnishingstatus,3


In [96]:
#размер изначального датафрейма (пока без изменения типов) - 55.5+ KB
df_pd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   price             545 non-null    int64 
 1   area              545 non-null    int64 
 2   bedrooms          545 non-null    int64 
 3   bathrooms         545 non-null    int64 
 4   stories           545 non-null    int64 
 5   mainroad          545 non-null    object
 6   guestroom         545 non-null    object
 7   basement          545 non-null    object
 8   hotwaterheating   545 non-null    object
 9   airconditioning   545 non-null    object
 10  parking           545 non-null    int64 
 11  prefarea          545 non-null    object
 12  furnishingstatus  545 non-null    object
dtypes: int64(6), object(7)
memory usage: 55.5+ KB


In [97]:
#создаём копию датафрейма, в котором приведём значения к выбранным типам
df_pd_optimized = df_pd
df_pd_optimized['price'] = df_pd_optimized['price'].astype(np.int32)
df_pd_optimized['area'] = df_pd_optimized['area'].astype(np.int16)
df_pd_optimized['bedrooms'] = df_pd_optimized['bedrooms'].astype(np.int8)
df_pd_optimized['bathrooms'] = df_pd_optimized['bathrooms'].astype(np.int8)
df_pd_optimized['stories'] = df_pd_optimized['stories'].astype(np.int8)
df_pd_optimized['parking'] = df_pd_optimized['parking'].astype(np.int8)

In [99]:
#смотрим промежуточный размер (пока с изменением только по int) - 35.3+ KB, т.е. уменьшили объём на 36%
df_pd_optimized.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   price             545 non-null    int32 
 1   area              545 non-null    int16 
 2   bedrooms          545 non-null    int8  
 3   bathrooms         545 non-null    int8  
 4   stories           545 non-null    int8  
 5   mainroad          545 non-null    object
 6   guestroom         545 non-null    object
 7   basement          545 non-null    object
 8   hotwaterheating   545 non-null    object
 9   airconditioning   545 non-null    object
 10  parking           545 non-null    int8  
 11  prefarea          545 non-null    object
 12  furnishingstatus  545 non-null    object
dtypes: int16(1), int32(1), int8(4), object(7)
memory usage: 35.3+ KB


In [101]:
#продолжаем оптимизацию (по object)
df_pd_optimized['mainroad'] = df_pd_optimized['mainroad'].astype("category")
df_pd_optimized['guestroom'] = df_pd_optimized['guestroom'].astype("category")
df_pd_optimized['basement'] = df_pd_optimized['basement'].astype("category")
df_pd_optimized['hotwaterheating'] = df_pd_optimized['hotwaterheating'].astype("category")
df_pd_optimized['airconditioning'] = df_pd_optimized['airconditioning'].astype("category")
df_pd_optimized['prefarea'] = df_pd_optimized['prefarea'].astype("category")
df_pd_optimized['furnishingstatus'] = df_pd_optimized['furnishingstatus'].astype("category")

In [102]:
#смотрим итоговый размер - 10.0 KB, от изначального объёма уменьшение на 82% (круто)
df_pd_optimized.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   price             545 non-null    int32   
 1   area              545 non-null    int16   
 2   bedrooms          545 non-null    int8    
 3   bathrooms         545 non-null    int8    
 4   stories           545 non-null    int8    
 5   mainroad          545 non-null    category
 6   guestroom         545 non-null    category
 7   basement          545 non-null    category
 8   hotwaterheating   545 non-null    category
 9   airconditioning   545 non-null    category
 10  parking           545 non-null    int8    
 11  prefarea          545 non-null    category
 12  furnishingstatus  545 non-null    category
dtypes: category(7), int16(1), int32(1), int8(4)
memory usage: 10.0 KB
